# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [19]:
import duckdb
from getpass import getpass

con = duckdb.connect()

# Token entered securely, not pasted in the cell — this repo is public
hf_token = getpass("Paste your Hugging Face READ token: ")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"

# Point directly at the month=2026-03 partition rather than scanning the full 79M-row table
month_path = f"{base}/fact_content_daily_performance/month=2026-03/data_0.parquet"

Paste your Hugging Face READ token: ··········


In [20]:
# Rebuild label_df from w03_data_contract.ipynb
label_df = con.sql(f"""
    WITH halves AS (
        SELECT
            content_hash_id,
            SUM(CASE WHEN report_date < '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_first_half,
            SUM(CASE WHEN report_date >= '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_second_half
        FROM read_parquet('{month_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        content_hash_id,
        impr_first_half,
        impr_second_half,
        CASE WHEN impr_second_half < impr_first_half THEN 1 ELSE 0 END AS is_declining_proxy
    FROM halves
    WHERE impr_first_half > 0
""").df()

print(f"Rows: {len(label_df)}")
print(f"Declining rate: {label_df['is_declining_proxy'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 151981
Declining rate: 0.438


## 0.5 Signal Experiments

In [21]:
# Signal 1: CTR-vs-position
pf = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

print(pf.shape)
print("rows where avg_position is null (page had ZERO real-position days all month):", pf["avg_position"].isna().sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 4)
rows where avg_position is null (page had ZERO real-position days all month): 1434


In [22]:
pf_valid = pf.dropna(subset=["avg_position"]).copy()
pf_valid["eligible"] = pf_valid["total_impressions"] >= 10
print("eligible:", pf_valid['eligible'].sum(), "/ excluded:", (~pf_valid['eligible']).sum())

pf_valid["zero_clicks_at_position"] = (
    (pf_valid["avg_position"] <= 10) & (pf_valid["total_clicks"] == 0) & (pf_valid["eligible"])
).astype(int)

good_pos_gated = pf_valid[(pf_valid["avg_position"] <= 10) & (pf_valid["eligible"])].copy()
good_pos_gated["zero_clicks"] = (good_pos_gated["total_clicks"] == 0).astype(int)
sig1_gated = good_pos_gated.merge(label_df, on="content_hash_id")
print(sig1_gated.groupby("zero_clicks").agg(
    n=("is_declining_proxy", "size"),
    pct_declining=("is_declining_proxy", "mean")
).round(3))

eligible: 143183 / excluded: 32121
                 n  pct_declining
zero_clicks                      
0            41217          0.365
1            25188          0.483


- 36.5% vs 48.3% declining, n = 41,217 / 25,188, 11.8-point gap
  - Note: A bit weaker than a non-gated version (was 14.4-point gap, but gating clears some noise at the cost of catching some legitimate data in the crossfire)
- **CONFIRMED**

In [23]:
# Signal 2: engagement-vs-volume
vol = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(ga4_sessions) AS total_sessions
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

print(vol.shape)
print(vol[["total_impressions", "total_sessions"]].describe())

vol["session_rate"] = vol["total_sessions"] / vol["total_impressions"]
print(vol["session_rate"].describe())
print((vol["total_sessions"] == 0).mean())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(63856, 3)
       total_impressions  total_sessions
count       63856.000000    63856.000000
mean         1336.393135       19.399195
std          5549.319325       54.752403
min             1.000000        0.000000
25%            17.000000        1.000000
50%           106.000000        4.000000
75%           694.000000       15.000000
max        617124.000000     2603.000000
count    63856.000000
mean         0.169645
std          0.362552
min          0.000000
25%          0.014493
50%          0.040909
75%          0.146341
max         12.000000
Name: session_rate, dtype: float64
0.0032260085191681285


In [24]:
median_rate = vol["session_rate"].median()
vol["engagement_weak"] = (vol["session_rate"] < median_rate).astype(int)

sig2 = vol.merge(label_df, on="content_hash_id")
print(sig2.groupby("engagement_weak").agg(
    n=("is_declining_proxy", "size"),
    pct_declining=("is_declining_proxy", "mean")
).round(3))

                     n  pct_declining
engagement_weak                      
0                29246          0.439
1                29764          0.354


In [25]:
# Does total_impressions alone (nothing to do with sessions) already predict decline in the same direction as engagement_weak did?
vol_labeled = vol.merge(label_df, on="content_hash_id")

median_impr = vol_labeled["total_impressions"].median()
vol_labeled["low_impressions"] = (vol_labeled["total_impressions"] < median_impr).astype(int)

print(vol_labeled.groupby("low_impressions").agg(
    n=("is_declining_proxy", "size"),
    pct_declining=("is_declining_proxy", "mean")
).round(3))

print("\ncorrelation between session_rate and total_impressions:",
      vol_labeled["session_rate"].corr(vol_labeled["total_impressions"]))

                     n  pct_declining
low_impressions                      
0                29557          0.366
1                29453          0.426

correlation between session_rate and total_impressions: -0.10344383452384948


Signal 2: weak engagement (session_rate below median) vs. decline
- n=29,246 above median (43.9% declining) vs. n=29,764 below median (35.4% declining)
- **OPPOSITE**
  - Partially explained by a mechanical link — low-impression pages inflate session_rate and independently decline more often. Overlap likely accounts for some but not all of the gap; the data doesn't cleanly separate a volume effect from a genuine engagement effect here.
  - Either way, session_rate isn't a safe input for the score — it's confounded with a variable (impression volume) that has its own independent relationship to the label.

In [26]:
# Signal 3: position trend, leakage-safe (week 1 vs week 2 of the first half only)
postrend = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN report_date < '2026-03-08' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_wk1,
        AVG(CASE WHEN report_date >= '2026-03-08' AND report_date < '2026-03-16' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_wk2,
        SUM(gsc_impressions) AS total_impressions
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE AND report_date < '2026-03-16'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

print(postrend.shape)

postrend = postrend.dropna(subset=["avg_position_wk1", "avg_position_wk2"])
print("rows with position data in both weeks:", len(postrend))

postrend["position_change"] = postrend["avg_position_wk2"] - postrend["avg_position_wk1"]
postrend["position_worsened"] = (postrend["position_change"] > 0).astype(int)

# Same evidence-floor gate as the CTR signal (full-month impressions >= 10),
# applied by EXCLUDING ineligible rows -- not recoding them into "not worsening."
postrend_eligible = postrend.merge(pf_valid[["content_hash_id", "eligible"]], on="content_hash_id", how="left")
postrend_eligible = postrend_eligible[postrend_eligible["eligible"] == True].copy()
print("eligible rows in position-trend check:", len(postrend_eligible))

sig3 = postrend_eligible.merge(label_df, on="content_hash_id")
print(sig3.groupby("position_worsened").agg(
    n=("is_declining_proxy", "size"),
    pct_declining=("is_declining_proxy", "mean")
).round(3))

(151981, 4)
rows with position data in both weeks: 123952
eligible rows in position-trend check: 119176
                       n  pct_declining
position_worsened                      
0                  55814          0.415
1                  63362          0.457


- 4.2-point gap (41.5% vs 45.7%)
- direction is right (worsening position associates with more decline), but noticeably weaker than Signal 1's 14-point gap on similar-sized n.
- **MIXED (Weak but real)**

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: A page is worth reviewing if it's ranking well but getting zero clicks, or if it's position is slipping week over week. If neither flag appears, leave it alone.

Reason codes:
- zero_clicks_at_position → refresh_and_review_ctr (strong signal, priority)
- position_worsened → refresh (weaker signal, lighter action)
- zero_clicks_at_position AND position_worsened → refresh_and_review_ctr (strongest case takes priority)
- no_flag → monitor

Note: Signals were re-verified after finding and fixing a shared position-averaging bug, and neither verdict moved

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Signal Checks run above in 0.5

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import os

scored = pf_valid.merge(
    postrend_eligible[["content_hash_id", "position_worsened"]],
    on="content_hash_id", how="left"
)
scored["position_worsened"] = scored["position_worsened"].fillna(0).astype(int)

scored["score"] = scored["zero_clicks_at_position"] * 2 + scored["position_worsened"] * 1

conditions = [
    (scored["zero_clicks_at_position"] == 1) & (scored["position_worsened"] == 1),
    (scored["zero_clicks_at_position"] == 1),
    (scored["position_worsened"] == 1),
]
reason_codes = ["zero_clicks_and_position_worsened", "zero_clicks_at_position", "position_worsened"]
actions      = ["refresh_and_review_ctr", "refresh_and_review_ctr", "refresh"]

scored["reason_code"] = np.select(conditions, reason_codes, default="no_flag")
scored["action"]      = np.select(conditions, actions, default="monitor")

scored = scored.sort_values(["score", "total_impressions"], ascending=[False, False]).reset_index(drop=True)
scored["rank"] = scored.index + 1

print("Rows excluded for missing position data:", len(pf) - len(pf_valid))
print("\nScore distribution:")
print(scored["score"].value_counts().sort_index())
print("\nAction distribution:")
print(scored["action"].value_counts())
print("\nReason code distribution:")
print(scored["reason_code"].value_counts())
print(f"\nTotal rows: {len(scored)}")

os.makedirs("work/outputs", exist_ok=True)
output_cols = ["content_hash_id", "rank", "score", "reason_code", "action",
               "avg_position", "total_impressions", "total_clicks", "eligible", "position_worsened"]
scored[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Written: work/outputs/baseline_action_score.csv")

Rows excluded for missing position data: 1434

Score distribution:
score
0    93762
1    52753
2    18180
3    10609
Name: count, dtype: int64

Action distribution:
action
monitor                   93762
refresh                   52753
refresh_and_review_ctr    28789
Name: count, dtype: int64

Reason code distribution:
reason_code
no_flag                              93762
position_worsened                    52753
zero_clicks_at_position              18180
zero_clicks_and_position_worsened    10609
Name: count, dtype: int64

Total rows: 175304
Written: work/outputs/baseline_action_score.csv


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top10 = scored.head(10)[["rank", "content_hash_id", "score", "reason_code", "action",
                          "avg_position", "total_impressions", "total_clicks", "position_worsened"]]
print(top10.to_string(index=False))

 rank          content_hash_id  score                       reason_code                 action  avg_position  total_impressions  total_clicks  position_worsened
    1 content_0c5606abaaab3178      3 zero_clicks_and_position_worsened refresh_and_review_ctr      5.694764            38865.0           0.0                  1
    2 content_23a42776a7009b65      3 zero_clicks_and_position_worsened refresh_and_review_ctr      9.419929            28950.0           0.0                  1
    3 content_c9f840183215651b      3 zero_clicks_and_position_worsened refresh_and_review_ctr      9.092933            21519.0           0.0                  1
    4 content_93d76695da196fdf      3 zero_clicks_and_position_worsened refresh_and_review_ctr      8.053905            19292.0           0.0                  1
    5 content_b9d46abd9ffa6c8a      3 zero_clicks_and_position_worsened refresh_and_review_ctr      7.973915            15902.0           0.0                  1
    6 content_d2eb49b1f5f3fa34    

In [30]:
for code in ["zero_clicks_at_position", "position_worsened", "no_flag"]:
    sample = scored[scored["reason_code"] == code].sort_values("total_impressions", ascending=False).head(1)
    print(sample[["content_hash_id", "reason_code", "action", "avg_position", "total_impressions", "total_clicks", "position_worsened"]].to_string(index=False))
    print()

         content_hash_id             reason_code                 action  avg_position  total_impressions  total_clicks  position_worsened
content_bf078007df823490 zero_clicks_at_position refresh_and_review_ctr      7.906249            44707.0           0.0                  0

         content_hash_id       reason_code  action  avg_position  total_impressions  total_clicks  position_worsened
content_ec2e0346994fb5a5 position_worsened refresh      2.854514           245276.0        1480.0                  1

         content_hash_id reason_code  action  avg_position  total_impressions  total_clicks  position_worsened
content_eadb33b5df496f4a     no_flag monitor      2.383011           617124.0        5668.0                  0



**rank | action | reason_code | what would make it wrong**

**1.** refresh_and_review_ctr **|** zero_clicks_and_position_worsened **|** At this volume, literal zero clicks all month is unusual enough to suspect a tracking break (canonical mismatch, GSC property misconfiguration) rather than pure content weakness

**2.** refresh_and_review_ctr **|** zero_clicks_and_position_worsened **|** Same tracking-artifact risk as row 1, slightly weaker position makes a real snippet/title problem somewhat more plausible

**3.** refresh_and_review_ctr **|** zero_clicks_and_position_worsened **|** Same caveat as row 2

**4.** refresh_and_review_ctr **|** zero_clicks_and_position_worsened **|** Same tracking-artifact caution

**5.** refresh_and_review_ctr **|** zero_clicks_and_position_worsened **|** Same caveat

**6.** refresh_and_review_ctr **|** zero_clicks_and_position_worsened **|** Same caveat

**7.** refresh_and_review_ctr **|** zero_clicks_and_position_worsened **|** Ranking #2 on average with thousands of impressions and zero clicks all month is the strongest case for a measurement bug. A featured snippet siphoning clicks, or a tracking mismatch, could produce this legitimately

**8.** refresh_and_review_ctr **|** zero_clicks_and_position_worsened **|** Same tracking-artifact caution given the strong position

**9.** refresh_and_review_ctr **|** zero_clicks_and_position_worsened **|** Same caveat

**10.** refresh_and_review_ctr **|** zero_clicks_and_position_worsened **|** Same caveat, lowest-volume of the ten but still well above the eligibility floor

# Supplementary Samples from other reason codes

**reason code | action | what would make it wrong**

zero_clicks_at_position **|** refresh_and_review_ctr **|** Same tracking-artifact concern as the top 10 — at this volume, literal zero clicks with a steady position is arguably more suspicious than the double-flagged cases, since there's no other explanation (like slipping rank) to point to.

position_worsened **|** refresh **|** Could be a seasonal or one-off ranking dip rather than a genuine decline. Still a good illustration of why refresh (lighter action) is right here and refresh_and_review_ctr would be wrong — the page is clearly still earning legitimate traffic and clicks.

no_flag **|** monitor **|** Nothing here suggests this needs review. Would only be wrong if this page had some upcoming risk not visible in the current window (such as a competitor starting to outranking it) outside what any single-month rule can see

## Testing For Outlier Similarities

In [31]:
weak_picks = ["content_d397987113cb84a0", "content_bf078007df823490"]  # rank 7 top-10, and the zero_clicks_at_position spot-check

# Step 1: client_hash_id for both
weak_clients = con.sql(f"""
    SELECT DISTINCT content_hash_id, client_hash_id
    FROM read_parquet('{month_path}')
    WHERE content_hash_id IN ({', '.join(f"'{c}'" for c in weak_picks)})
""").df()
print(weak_clients)
print()

# Step 2: rebuild client-level zero-click rate at the FINAL rule's own cutoffs (position<=10, impressions>=10)
client_rates_final = con.sql(f"""
    SELECT
        client_hash_id,
        COUNT(*) AS n_pages_eligible,
        AVG(CASE WHEN total_clicks = 0 THEN 1.0 ELSE 0 END) AS pct_zero_click
    FROM (
        SELECT
            content_hash_id, client_hash_id,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks
        FROM read_parquet('{month_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
        HAVING SUM(gsc_impressions) >= 10
    )
    WHERE avg_position <= 10
    GROUP BY client_hash_id
    HAVING n_pages_eligible >= 20
""").df()
overall_rate = client_rates_final["pct_zero_click"].mean()
print(f"Overall zero-click rate at final cutoffs: {overall_rate:.3f}")

compare = weak_clients.merge(client_rates_final, on="client_hash_id", how="left")
print(compare)
print()

# Step 3: check dim_content for shared content characteristics
content_check = con.sql(f"""
    SELECT content_hash_id, content_type, word_count, provider_used, model_used
    FROM read_parquet('{base}/dim_content.parquet')
    WHERE content_hash_id IN ({', '.join(f"'{c}'" for c in weak_picks)})
""").df()
print(content_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id           client_hash_id
0  content_d397987113cb84a0  client_73cda7b4e4f265ea
1  content_bf078007df823490  client_23a62021009f63c4

Overall zero-click rate at final cutoffs: 0.584
            content_hash_id           client_hash_id  n_pages_eligible  \
0  content_d397987113cb84a0  client_73cda7b4e4f265ea             15871   
1  content_bf078007df823490  client_23a62021009f63c4              3156   

   pct_zero_click  
0        0.259467  
1        0.393853  



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id     content_type  word_count provider_used  \
0  content_bf078007df823490  keyword article        3318        google   
1  content_d397987113cb84a0  keyword article        <NA>          None   

               model_used  
0  gemini-3-flash-preview  
1                    None  


In [32]:
# Correct pooled rate: total zero-click pages / total eligible pages, not a mean-of-means
total_eligible = client_rates_final["n_pages_eligible"].sum()
total_zero = (client_rates_final["pct_zero_click"] * client_rates_final["n_pages_eligible"]).sum()
pooled_rate = total_zero / total_eligible
print(f"Correct pooled zero-click rate: {pooled_rate:.3f}")  # should land near 0.379

# Where do our two clients sit in the actual distribution, by percentile rank?
client_rates_final["percentile_rank"] = client_rates_final["pct_zero_click"].rank(pct=True)
compare_fixed = weak_clients.merge(client_rates_final, on="client_hash_id", how="left")
print(compare_fixed[["content_hash_id", "client_hash_id", "n_pages_eligible", "pct_zero_click", "percentile_rank"]])

Correct pooled zero-click rate: 0.398
            content_hash_id           client_hash_id  n_pages_eligible  \
0  content_d397987113cb84a0  client_73cda7b4e4f265ea             15871   
1  content_bf078007df823490  client_23a62021009f63c4              3156   

   pct_zero_click  percentile_rank  
0        0.259467         0.081081  
1        0.393853         0.297297  


- Tracking Bug theory less credible. Both clients sit below the population's typical zero-click rate, meaning their other pages click just fine, some considerably better than average.
- Probably isolated anomalies
- Could still be tracking issues, just isolated page-specific issues instead of a broad client-wide problem

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

#### Weak picks

Two of the flagged rows deserve explicit scrutiny rather than being taken at face value: content_d397987113cb84a0 (top-10, rank 7 — position ~1.95, 9,887 impressions, zero clicks) and content_bf078007df823490 (highest-volume single-flag case — position ~7.9, 44,707 impressions, zero clicks). Both are page-one pages with substantial traffic and literally zero recorded clicks in the window, which is the profile most likely to reflect a measurement problem rather than a genuine content problem.

Checked whether these cluster at the client level, which would point to a systemic tracking failure, which they don't. Both clients sit well below their peer group's typical zero-click rate (8th and 30th percentile respectively), meaning each client's other pages convert normally. That rules out a client-wide integration issue as the explanation, but doesn't resolve whether either specific page has a genuine content problem or an isolated, page-level tracking issue.

This is a real limitation of the rule: zero_clicks_at_position can't distinguish "nobody clicks because the content doesn't earn the click" from "nobody clicks because the click isn't being recorded." Both produce the identical signature in this data. A human reviewer checking the live page is the only way to resolve it — which is exactly what routing these to refresh_and_review_ctr for manual review is for, rather than auto-actioning them.

A third possible explanation was considered, that being AI Overviews answering the query directly in the SERP, suppressing the click regardless of page quality. This isn't testable with this dataset — no SERP-feature column exists in the warehouse schema. A related, testable question — whether these pages are compensating with real traffic from AI chat tools (ChatGPT, Perplexity, etc., via sessions_ai) — was checked directly and doesn't hold up. AI-referral sessions are near-zero and, if anything, slightly lower for the zero-click group (2.7% of pages) than the clicked group (3.5%), the opposite of what the substitution theory would predict. This leaves the tracking-artifact and genuine-content-problem explanations from earlier as the remaining candidates, with no evidence pointing toward AI Overview substitution as a driver here — though its absence from this schema means it can't be fully excluded, only that the adjacent signal available doesn't support it.

In [33]:
ai_check = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        SUM(sessions_ai) AS total_ai_sessions,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 10
""").df()

ai_check = ai_check.dropna(subset=["avg_position"])
ai_check["zero_gsc_clicks"] = (ai_check["total_clicks"] == 0).astype(int)
decent = ai_check[ai_check["avg_position"] <= 10]

print(decent.groupby("zero_gsc_clicks")["total_ai_sessions"].agg(["count", "mean", "median", lambda x: (x>0).mean()]).round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                 count   mean  median  <lambda_0>
zero_gsc_clicks                                  
0                22545  0.070     0.0       0.035
1                 6590  0.049     0.0       0.027


#### Leakage Check

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm no label-derived or future-window inputs made it into the score

feature_cols = ["avg_position", "total_impressions", "total_clicks", "eligible",
                 "zero_clicks_at_position", "position_worsened", "score"]

banned = ["is_declining_proxy", "impr_first_half", "impr_second_half"]

print("Banned columns present in scored feature set:",
      [c for c in banned if c in scored.columns])

# Confirm position_worsened's own inputs never touch the label's window (report_date >= '2026-03-16')
print("\npostrend built from report_date < '2026-03-16' only — confirmed in query WHERE clause")
print("label_df built from full March, split at '2026-03-16' — confirmed in query CASE WHEN")
print("\nNo overlap: postrend's week1/week2 split (days 1-7, 8-15) sits entirely")
print("before label_df's window even starts (day 16 onward).")

# Sanity check: scored's row count should match pf_valid, not include is_declining_proxy at all
print("\nscored columns:", list(scored.columns))
assert "is_declining_proxy" not in scored.columns, "LEAKAGE: label column present in scored features"
print("\nPASS: label column not present in scored feature/output table.")

Banned columns present in scored feature set: []

postrend built from report_date < '2026-03-16' only — confirmed in query WHERE clause
label_df built from full March, split at '2026-03-16' — confirmed in query CASE WHEN

No overlap: postrend's week1/week2 split (days 1-7, 8-15) sits entirely
before label_df's window even starts (day 16 onward).

scored columns: ['content_hash_id', 'avg_position', 'total_impressions', 'total_clicks', 'eligible', 'zero_clicks_at_position', 'position_worsened', 'score', 'reason_code', 'action', 'rank']

PASS: label column not present in scored feature/output table.


In [35]:
low_evidence_slippers = scored[
    (scored["position_worsened"] == 1) & (scored["total_impressions"] < 10)
]
print(f"position_worsened=1 rows with full-month impressions < 10: {len(low_evidence_slippers)}")
print(f"as a share of all position_worsened=1 rows: {len(low_evidence_slippers) / (scored['position_worsened']==1).sum():.1%}")

position_worsened=1 rows with full-month impressions < 10: 0
as a share of all position_worsened=1 rows: 0.0%


Confirmed no future-window or label-derived inputs reached the score. zero_clicks_at_position uses only avg_position and total_clicks, aggregated across the full month — knowable at any point once the month's data lands, not dependent on the half-split used for the label. position_worsened uses only days 1–15 (week 1 vs. week 2), entirely before the label's own window starts on day 16 — the same leakage-safe boundary established when this signal was first built. is_declining_proxy, impr_first_half, and impr_second_half never appear in scored's columns; they exist only in label_df, used solely to validate signals during the check phase, never as score inputs.

In [36]:
scored_labeled = scored.merge(label_df[["content_hash_id", "is_declining_proxy"]], on="content_hash_id")
scored_labeled = scored_labeled.sort_values("rank")

for k in [20, 50, 100, 200]:
    top_k = scored_labeled.head(k)
    precision = top_k["is_declining_proxy"].mean()
    print(f"precision@{k}: {precision:.3f}  ({top_k['is_declining_proxy'].sum()}/{k} declining)")

precision@20: 0.600  (12/20 declining)
precision@50: 0.560  (28/50 declining)
precision@100: 0.510  (51/100 declining)
precision@200: 0.505  (101/200 declining)


Rule is mediocre at best, with accuracy being a 50/50 split on larger K values.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.